# Semantic Search RAG vs. Moment RAG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/calvnguyen/semantic_search_moment_rag/blob/main/moment_rag.ipynb)

**Module 3 (Agentic RAG) project.** We build two retrieval-augmented pipelines over the
transcript of a single YouTube video and compare them:

- **Part 1 — Baseline semantic-search RAG:** fixed-size chunks → embeddings → cosine top-k → answer.
- **Part 2 — Moment RAG:** timestamped *moments* → ingestion enrichment (HyDE-style hypothetical
  questions) → query **decomposition** → **hybrid** retrieval (dense + BM25, RRF-fused) →
  **cross-encoder re-rank** → moment-level rollup with **click-to-timestamp citations**.
- **Part 3 — Comparison & analysis.**

The Moment RAG design mirrors the `Moment_RAG/` reference codebase from this module — same ideas
(query-side intelligence, HyDE-at-ingestion, RRF fusion, cross-encoder rerank, moment rollup),
on a lighter, fully transparent stack (numpy + `rank-bm25` + a small cross-encoder) instead of
the Qdrant/fastembed setup, so every step is readable here.

> **Run on Colab:** click the badge above, then **Runtime → Run all**. The first cell installs
> dependencies and prompts for your `OPENAI_API_KEY`. Running locally instead? Follow the README
> (`.venv` + `.env`) and the same cell becomes a no-op.

> **Video:** *What is a Vector Database? Powering Semantic Search & AI Applications* — https://www.youtube.com/watch?v=gl1r1XV0SLw


In [1]:
# === Setup — runs anywhere (Google Colab OR a local venv) ===
# On Colab: installs dependencies, pulls the cached transcript, and prompts for your key.
# Locally (with the README's .venv + .env): detects everything is present and does nothing.
import importlib.util, os, pathlib, subprocess, sys, urllib.request

_need = [m for m in ("openai", "numpy", "tiktoken", "rank_bm25", "sentence_transformers",
                     "youtube_transcript_api", "dotenv", "chromadb")
         if importlib.util.find_spec(m) is None]
if _need:
    _pip = {"dotenv": "python-dotenv", "rank_bm25": "rank-bm25",
            "youtube_transcript_api": "youtube-transcript-api",
            "sentence_transformers": "sentence-transformers"}
    print("installing:", ", ".join(_need))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *[_pip.get(m, m) for m in _need]], check=True)

# YouTube often blocks transcript fetches from Colab IPs, so pull the cached transcript
# committed in the repo. (No-op locally, where data/transcript.json already exists.)
_d = pathlib.Path("data"); _d.mkdir(exist_ok=True)
_RAW = "https://raw.githubusercontent.com/calvnguyen/semantic_search_moment_rag/main/data/transcript.json"
if not (_d / "transcript.json").exists():
    try:
        urllib.request.urlretrieve(_RAW, _d / "transcript.json")
        print("fetched cached transcript from GitHub")
    except Exception as e:
        print("no cached transcript (will try YouTube live):", e)

# Key: load from .env locally, else prompt (Colab).
from dotenv import load_dotenv; load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY: ")
print("environment ready")

environment ready


## Section 0 — Setup & transcript

We load the OpenAI client, then fetch the video's captions and reshape them into timed **cues**
(`{start_ms, end_ms, text}`) — the same shape the reference codebase uses. The timestamp on each
cue is what later lets a retrieved *moment* deep-link back into the video.

In [2]:
import os, json, re, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()  # reads OPENAI_API_KEY from the environment / .env

VIDEO_ID    = "gl1r1XV0SLw"
EMBED_MODEL = "text-embedding-3-small"
LLM_MODEL   = "gpt-4o-mini"

DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
ENC = tiktoken.get_encoding("cl100k_base")
print("setup ok")

setup ok


In [3]:
def fetch_transcript(video_id: str) -> list[dict]:
    """Fetch captions and reshape to timed cues. Cached to data/transcript.json.

    Handles both the legacy static API (youtube-transcript-api <=0.6.x) and the
    newer instance API (>=1.0).
    """
    cache = DATA_DIR / "transcript.json"
    if cache.exists():
        return json.loads(cache.read_text())

    from youtube_transcript_api import YouTubeTranscriptApi
    if hasattr(YouTubeTranscriptApi, "get_transcript"):          # <=0.6.x
        raw = YouTubeTranscriptApi.get_transcript(video_id)
        segs = [(r["text"], r["start"], r["duration"]) for r in raw]
    else:                                                         # >=1.0
        fetched = YouTubeTranscriptApi().fetch(video_id)
        segs = [(s.text, s.start, s.duration) for s in fetched]

    cues = []
    for text, start, dur in segs:
        text = (text or "").replace("\n", " ").strip()
        if not text:
            continue
        cues.append({
            "start_ms": int(start * 1000),
            "end_ms":   int((start + dur) * 1000),
            "text":     text,
        })
    cache.write_text(json.dumps(cues, indent=2))
    return cues

cues = fetch_transcript(VIDEO_ID)
print(f"{len(cues)} cues, ~{sum(len(ENC.encode(c['text'])) for c in cues)} tokens total")
cues[:3]

109 cues, ~1512 tokens total


[{'start_ms': 30, 'end_ms': 1789, 'text': 'What is a vector database?'},
 {'start_ms': 2029,
  'end_ms': 3889,
  'text': 'Well, they say a picture is worth a thousand words.'},
 {'start_ms': 4450, 'end_ms': 5669, 'text': "So let's start with one."}]

In [4]:
def embed_texts(texts: list[str], model: str = EMBED_MODEL, batch: int = 256) -> np.ndarray:
    """Embed a list of texts with OpenAI, batched, returned as a float32 matrix."""
    out = []
    for i in range(0, len(texts), batch):
        resp = client.embeddings.create(model=model, input=texts[i:i + batch])
        out.extend(d.embedding for d in resp.data)
    return np.array(out, dtype=np.float32)

def cosine_topk(qvec: np.ndarray, matrix: np.ndarray, k: int) -> list[tuple[int, float]]:
    """Top-k rows of `matrix` by cosine similarity to `qvec`."""
    if len(matrix) == 0:
        return []
    q = qvec / (np.linalg.norm(qvec) + 1e-9)
    M = matrix / (np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-9)
    sims = M @ q
    idx = np.argsort(-sims)[:k]
    return [(int(i), float(sims[i])) for i in idx]

def fmt_ts(ms: int) -> str:
    s = int(ms / 1000); return f"{s // 60}:{s % 60:02d}"

def yt_link(ms: int) -> str:
    return f"https://www.youtube.com/watch?v={VIDEO_ID}&t={int(ms / 1000)}s"

print("helpers ready")

helpers ready


## Part 1 — Baseline semantic-search RAG

The classic pipeline: cut the transcript into **fixed-size token windows** (~256 tokens with 25%
overlap, mirroring the reference's `_window_chunks`), embed each, and retrieve the top-k by cosine
similarity. Chunk boundaries are arbitrary — they fall wherever the token budget runs out, not
where a topic actually starts or ends.

In [5]:
CHUNK_TOKENS = 256
OVERLAP = 0.25

def fixed_chunks(cues, chunk_tokens=CHUNK_TOKENS, overlap=OVERLAP):
    """Group consecutive cues into ~chunk_tokens windows with token overlap."""
    counts = [len(ENC.encode(c["text"])) for c in cues]
    stride_back = max(1, int(chunk_tokens * overlap))
    chunks, i, n = [], 0, len(cues)
    while i < n:
        tok, j = 0, i
        while j < n and tok + counts[j] <= chunk_tokens:
            tok += counts[j]; j += 1
        if j == i:
            j = i + 1
        window = cues[i:j]
        chunks.append({
            "chunk_id": f"c{len(chunks):04d}",
            "text": " ".join(c["text"] for c in window),
            "start_ms": window[0]["start_ms"],
            "end_ms": window[-1]["end_ms"],
            "n_tokens": tok,
        })
        if j >= n:
            break
        # step back a few cues so the next window overlaps by ~stride_back tokens
        back, t, k = 0, 0, j - 1
        while k >= i and t < stride_back:
            t += counts[k]; back += 1; k -= 1
        i = max(i + 1, j - back)
    return chunks

base_chunks = fixed_chunks(cues)
base_matrix = embed_texts([c["text"] for c in base_chunks])
print(f"{len(base_chunks)} fixed chunks, embedding matrix {base_matrix.shape}")

9 fixed chunks, embedding matrix (9, 1536)


In [6]:
def baseline_rag(query: str, k: int = 4) -> dict:
    qv = embed_texts([query])[0]
    hits = cosine_topk(qv, base_matrix, k)
    context = "\n\n".join(
        f"[{n+1}] {base_chunks[i]['text']}" for n, (i, _) in enumerate(hits)
    )
    prompt = (
        "Answer the question using ONLY the context below. "
        "If the answer isn't in the context, say so.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}"
    )
    resp = client.chat.completions.create(
        model=LLM_MODEL, temperature=0.2,
        messages=[{"role": "user", "content": prompt}],
    )
    return {
        "answer": resp.choices[0].message.content,
        "chunks": [{"start": fmt_ts(base_chunks[i]["start_ms"]), "score": round(s, 3),
                    "text": base_chunks[i]["text"][:160] + "…"} for i, s in hits],
    }

demo = baseline_rag("How does a vector database find similar vectors?")
print(demo["answer"]); print("\n--- retrieved chunks ---")
for c in demo["chunks"]:
    print(f'  [{c["start"]}] score={c["score"]}  {c["text"]}')

A vector database finds similar vectors by performing similarity searches as mathematical operations, looking for vector embeddings that are close to each other in vector space. It uses vector indexing with approximate nearest neighbor (ANN) algorithms to quickly find vectors that are very likely to be among the closest matches instead of finding the exact closest match.

--- retrieved chunks ---
  [7:53] score=0.732  you can't effectively and efficiently compare your query vector to every single vector in the database. It would just be too slow. So there is a process to do t…
  [8:57] score=0.67  Now, vector databases are a core feature of something called RAG, retrieval  augmented generation, where vector databases store chunks of documents and articles…
  [2:23] score=0.651  similar items are positioned close together in vector space and dissimilar items are positioned far apart, and with vector databases, we can perform similarity …
  [6:45] score=0.623  and then as we get to deepe

## Part 2 — Moment RAG

Now we move the intelligence to the **query side** and make retrieval land on *moments*.

**Ingestion (once):**
1. **Segment into moments** — group cues at semantic breakpoints (cosine distance between adjacent
   cues), so each moment is a coherent span with real `start_ms`/`end_ms`, not an arbitrary window.
2. **Enrich each moment** — one LLM call per moment produces `hypothetical_questions` it answers,
   `keywords`, and a one-line `gist`. (This is HyDE-at-ingestion: we index the *questions a moment
   answers*, so a user's question can match a stored question, not just the raw text.)
3. **Index** — embed moment text **and** the hypothetical questions; build a BM25 sparse index too.

**Query time:** decompose → hybrid retrieve (dense-text + dense-questions + BM25, **RRF-fused**) →
**cross-encoder re-rank** → cited synthesis with timestamp deep-links.

In [7]:
def segment_moments(cues, percentile=80, min_tokens=60, max_tokens=400):
    """Group cues into semantically coherent moments using embedding-distance breakpoints."""
    cue_vecs = embed_texts([c["text"] for c in cues])
    norm = cue_vecs / (np.linalg.norm(cue_vecs, axis=1, keepdims=True) + 1e-9)
    dists = [1 - float(norm[i] @ norm[i + 1]) for i in range(len(cues) - 1)]
    thresh = float(np.percentile(dists, percentile)) if dists else 1.0

    moments, cur, cur_tok = [], [0], len(ENC.encode(cues[0]["text"]))
    for i, d in enumerate(dists):
        nxt_tok = len(ENC.encode(cues[i + 1]["text"]))
        breaking = (d >= thresh and cur_tok >= min_tokens) or cur_tok >= max_tokens
        if breaking:
            moments.append(cur); cur, cur_tok = [i + 1], nxt_tok
        else:
            cur.append(i + 1); cur_tok += nxt_tok
    if cur:
        moments.append(cur)

    out = []
    for idxs in moments:
        window = [cues[j] for j in idxs]
        out.append({
            "moment_id": f"m{len(out):04d}",
            "text": " ".join(c["text"] for c in window),
            "start_ms": window[0]["start_ms"],
            "end_ms": window[-1]["end_ms"],
        })
    return out

moments = segment_moments(cues)
print(f"{len(moments)} moments (vs {len(base_chunks)} fixed chunks)")
for m in moments[:4]:
    print(f'  [{fmt_ts(m["start_ms"])}-{fmt_ts(m["end_ms"])}] {m["text"][:90]}…')

10 moments (vs 9 fixed chunks)
  [0:00-1:28] What is a vector database? Well, they say a picture is worth a thousand words. So let's st…
  [1:28-2:43] Those concepts aren't really represented very well in these structured fields and that dis…
  [2:43-3:24] Now we can represent all sorts of unstructured data in a vector database. What could we pu…
  [3:24-3:58] Well, I said there are arrays of numbers and there are arrays of numbers where each positi…


In [8]:
ENRICH_PROMPT = """You are indexing a segment of a YouTube explainer transcript for search.
Return STRICT JSON with exactly these keys:
- "hypothetical_questions": array of 2-4 specific questions a viewer could ask that THIS segment answers.
- "keywords": array of 3-8 salient concepts or entities.
- "gist": one concise sentence summarizing the segment.

Segment:
{text}"""

def enrich_moment(m: dict) -> dict:
    resp = client.chat.completions.create(
        model=LLM_MODEL, temperature=0.1,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": ENRICH_PROMPT.format(text=m["text"])}],
    )
    data = json.loads(resp.choices[0].message.content)
    m["hypothetical_questions"] = list(data.get("hypothetical_questions") or [])[:4]
    m["keywords"] = list(data.get("keywords") or [])
    m["gist"] = (data.get("gist") or "").strip()
    return m

# per-moment calls are independent -> run them concurrently
with ThreadPoolExecutor(max_workers=8) as ex:
    moments = list(ex.map(enrich_moment, moments))

print("enriched. example:")
print(json.dumps({k: moments[0][k] for k in ("gist", "keywords", "hypothetical_questions")}, indent=2))

enriched. example:
{
  "gist": "The segment explains how images and their metadata can be stored in a relational database, highlighting the limitations in capturing their semantic context for effective querying.",
  "keywords": [
    "vector database",
    "relational database",
    "image storage",
    "metadata",
    "tags",
    "semantic context",
    "querying images"
  ],
  "hypothetical_questions": [
    "What is a vector database and how does it differ from a relational database?",
    "How can I store images and their metadata in a database?",
    "What limitations do traditional databases have when storing images?",
    "How can I query for images with similar characteristics using a database?"
  ]
}


In [9]:
from rank_bm25 import BM25Okapi

def toks(s: str) -> list[str]:
    return re.findall(r"\w+", s.lower())

# Dense index over moment TEXT
moment_vecs = embed_texts([m["text"] for m in moments])

# Dense index over hypothetical QUESTIONS (each maps back to its moment) — HyDE branch
q_texts, q_owner = [], []
for mi, m in enumerate(moments):
    for q in m["hypothetical_questions"]:
        q_texts.append(q); q_owner.append(mi)
q_vecs = embed_texts(q_texts) if q_texts else np.zeros((0, moment_vecs.shape[1]), dtype=np.float32)

# Sparse BM25 index over moment text
bm25 = BM25Okapi([toks(m["text"]) for m in moments])

print(f"indexed: {len(moments)} moment vectors, {len(q_texts)} question vectors, BM25 ready")

indexed: 10 moment vectors, 39 question vectors, BM25 ready


In [10]:
def decompose(query: str) -> list[str]:
    """One question -> 2-4 retrieval sub-queries (atomic questions pass through)."""
    prompt = (
        "Break the user's question into 2-4 focused sub-queries for retrieval. "
        'Return JSON {"sub_queries": [...]}. If the question is already atomic, '
        "return it as the only element.\n\nQuestion: " + query
    )
    resp = client.chat.completions.create(
        model=LLM_MODEL, temperature=0.1,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}],
    )
    subs = json.loads(resp.choices[0].message.content).get("sub_queries") or [query]
    return [s for s in subs if s][:4] or [query]

def rrf_fuse(ranked_lists: list[list[int]], k: int = 60) -> list[tuple[int, float]]:
    """Reciprocal Rank Fusion across ranked moment-id lists."""
    scores: dict[int, float] = {}
    for lst in ranked_lists:
        for rank, mid in enumerate(lst):
            scores[mid] = scores.get(mid, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])

def moment_retrieve(query: str, k: int = 8):
    """Decompose, then per sub-query fuse dense-text + dense-questions + BM25 with RRF."""
    subs = decompose(query)
    ranked_lists = []
    for sq in subs:
        sv = embed_texts([sq])[0]
        ranked_lists.append([mid for mid, _ in cosine_topk(sv, moment_vecs, k)])      # dense text
        if len(q_vecs):                                                                # dense questions
            seen = []
            for qi, _ in cosine_topk(sv, q_vecs, k):
                mo = q_owner[qi]
                if mo not in seen:
                    seen.append(mo)
            ranked_lists.append(seen)
        bm_scores = bm25.get_scores(toks(sq))                                          # sparse BM25
        ranked_lists.append([int(i) for i in np.argsort(-bm_scores)[:k]])
    return subs, rrf_fuse(ranked_lists)

subs, fused = moment_retrieve("How does a vector database index embeddings and search them quickly?")
print("sub-queries:", subs)
print("top fused moments:", [(mid, round(s, 4)) for mid, s in fused[:5]])

sub-queries: ['What is a vector database and how does it work?', 'How are embeddings indexed in a vector database?', 'What techniques are used for fast searching in vector databases?']
top fused moments: [(9, 0.1447), (2, 0.1434), (1, 0.1425), (5, 0.1056), (0, 0.0943)]


In [11]:
USE_RERANK = True
_reranker = None

def get_reranker():
    global _reranker
    if _reranker is None:
        from sentence_transformers import CrossEncoder
        _reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")  # small CPU model
    return _reranker

def rerank(query: str, moment_ids: list[int], top_n: int = 6):
    """Cross-encoder re-scores candidate moments against the ORIGINAL query."""
    rr = get_reranker()
    scores = rr.predict([(query, moments[mid]["text"]) for mid in moment_ids])
    order = np.argsort(-scores)
    return [(moment_ids[i], float(scores[i])) for i in order[:top_n]]

print("reranker stage ready (USE_RERANK =", USE_RERANK, ")")

reranker stage ready (USE_RERANK = True )


In [12]:
def moment_rag(query: str, top_n: int = 6, use_rerank: bool | None = None) -> dict:
    use_rerank = USE_RERANK if use_rerank is None else use_rerank
    subs, fused = moment_retrieve(query)
    cand = [mid for mid, _ in fused[:20]]
    ranked = rerank(query, cand, top_n) if (use_rerank and cand) else fused[:top_n]
    chosen = [moments[mid] for mid, _ in ranked]

    context = "\n\n".join(
        f"[{n+1}] ({fmt_ts(m['start_ms'])}) {m['text']}" for n, m in enumerate(chosen)
    )
    prompt = (
        "Answer the question using ONLY the context below. Cite supporting moments "
        "inline as [n]. If the answer isn't in the context, say so.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}"
    )
    resp = client.chat.completions.create(
        model=LLM_MODEL, temperature=0.2,
        messages=[{"role": "user", "content": prompt}],
    )
    citations = [{
        "n": n + 1, "timestamp": fmt_ts(m["start_ms"]),
        "link": yt_link(m["start_ms"]), "gist": m.get("gist", ""),
    } for n, m in enumerate(chosen)]
    return {"answer": resp.choices[0].message.content,
            "sub_queries": subs, "citations": citations}

res = moment_rag("How does a vector database index embeddings and search them quickly?")
print("Sub-queries:", res["sub_queries"], "\n")
print(res["answer"], "\n\n--- citations ---")
for c in res["citations"]:
    print(f'  [{c["n"]}] {c["timestamp"]}  {c["link"]}\n        {c["gist"]}')

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Sub-queries: ['What is a vector database and how does it work?', 'How are embeddings indexed in a vector database?', 'What techniques are used for fast searching in vector databases?'] 

A vector database indexes embeddings and searches them quickly using vector indexing methods, specifically approximate nearest neighbor (ANN) algorithms. These algorithms do not find the exact closest match but instead quickly identify vectors that are very likely to be among the closest matches, which improves search speed while trading a small amount of accuracy for this efficiency [2]. 

--- citations ---
  [1] 8:26  https://www.youtube.com/watch?v=gl1r1XV0SLw&t=506s
        This segment explains various indexing methods for vector databases, their impact on search speed, and their role in retrieval augmented generation.
  [2] 7:53  https://www.youtube.com/watch?v=gl1r1XV0SLw&t=473s
        Vector indexing utilizes approximate nearest neighbor algorithms to efficiently find likely closest matches fo

## Part 3 — Comparison & analysis

We run a shared query set through both pipelines, including a deliberately **multi-faceted**
question that the baseline tends to answer only half of.

In [13]:
QUERIES = [
    "How does a vector database find similar vectors?",
    "What is an embedding?",
    # multi-faceted: exposes the value of query decomposition
    "How does a vector database index embeddings, and how does that differ from a traditional database query?",
]

for q in QUERIES:
    print("=" * 100)
    print("Q:", q)
    b = baseline_rag(q)
    m = moment_rag(q)
    print("\n--- BASELINE ---\n", b["answer"])
    print("\n--- MOMENT RAG ---")
    print("sub-queries:", m["sub_queries"])
    print(m["answer"])
    print("citations:", [f'{c["timestamp"]} -> {c["link"]}' for c in m["citations"]])
    print()

Q: How does a vector database find similar vectors?



--- BASELINE ---
 A vector database finds similar vectors by performing similarity searches as mathematical operations, looking for vector embeddings that are close to each other in vector space. It uses vector indexing and approximate nearest neighbor (ANN) algorithms to quickly find vectors that are very likely to be among the closest matches, rather than finding the exact closest match.

--- MOMENT RAG ---
sub-queries: ['What algorithms do vector databases use to find similar vectors?', 'What is the role of distance metrics in vector similarity search?', 'How do indexing techniques improve the efficiency of vector similarity searches?']
A vector database finds similar vectors by using vector indexing and approximate nearest neighbor (ANN) algorithms. These methods allow the system to quickly find vectors that are very likely to be among the closest matches to a query vector, rather than comparing the query vector to every single vector in the database, which would be too slow [3]. 


--- BASELINE ---
 The context does not explicitly define what an embedding is.

--- MOMENT RAG ---
sub-queries: ['What is the definition of an embedding?', 'What are the applications of embeddings in machine learning?', 'How are embeddings created or generated?']
An embedding is essentially an array of numbers that captures the semantic essence of data, where similar items are positioned close together in vector space and dissimilar items are positioned far apart. These embeddings are created through embedding models that process data and extract progressively more abstract features, resulting in high-dimensional vectors that represent the essential characteristics of the input data [4][3].
citations: ['3:24 -> https://www.youtube.com/watch?v=gl1r1XV0SLw&t=204s', '5:14 -> https://www.youtube.com/watch?v=gl1r1XV0SLw&t=314s', '6:21 -> https://www.youtube.com/watch?v=gl1r1XV0SLw&t=381s', '1:28 -> https://www.youtube.com/watch?v=gl1r1XV0SLw&t=88s', '2:43 -> https://www.youtube.com/watch?v


--- BASELINE ---
 A vector database indexes embeddings using vector indexing, which employs approximate nearest neighbor (ANN) algorithms. These algorithms quickly find vectors that are likely to be among the closest matches instead of finding the exact closest match. This process improves search speed while trading a small amount of accuracy.

In contrast, a traditional database query, such as a simple selection based on specific attributes (e.g., "select star where color equals orange"), falls short because it does not capture the nuanced, multi-dimensional nature of unstructured data. Traditional queries rely on structured fields and do not effectively represent the semantic essence of the data, whereas vector databases represent data as mathematical vector embeddings that allow for similarity searches based on the proximity of vectors in vector space.

--- MOMENT RAG ---
sub-queries: ['How does a vector database index embeddings?', 'What are the differences between vector database

In [14]:
# Compact side-by-side summary
def unit_summary():
    rows = [
        ("Retrieval unit", f"{len(base_chunks)} fixed ~{CHUNK_TOKENS}-token chunks",
         f"{len(moments)} semantic moments"),
        ("Boundaries", "arbitrary (token budget)", "topic shifts (semantic breakpoints)"),
        ("Query handling", "single embedding lookup", "decompose -> hybrid (dense+BM25) -> RRF -> rerank"),
        ("Extra index signals", "none", f"{len(q_texts)} hypothetical-question vectors (HyDE)"),
        ("Citations", "chunk text only", "moment + clickable &t= timestamp deep-link"),
    ]
    w = max(len(r[0]) for r in rows)
    print(f'{"":<{w}} | {"BASELINE":<42} | MOMENT RAG')
    print("-" * 110)
    for label, a, b in rows:
        print(f"{label:<{w}} | {a:<42} | {b}")

unit_summary()

                    | BASELINE                                   | MOMENT RAG
--------------------------------------------------------------------------------------------------------------
Retrieval unit      | 9 fixed ~256-token chunks                  | 10 semantic moments
Boundaries          | arbitrary (token budget)                   | topic shifts (semantic breakpoints)
Query handling      | single embedding lookup                    | decompose -> hybrid (dense+BM25) -> RRF -> rerank
Extra index signals | none                                       | 39 hypothetical-question vectors (HyDE)
Citations           | chunk text only                            | moment + clickable &t= timestamp deep-link


### How Moment RAG changes answer quality

- **Citations land on the exact moment.** Every Moment RAG citation carries a real timestamp and a
  `&t=<sec>s` deep-link — you can jump to the spot in the video. Baseline chunks have no reliable,
  topic-aligned timestamp to cite.
- **Decomposition catches multi-part questions.** The multi-faceted query is split into sub-queries,
  so retrieval covers *both* facets (indexing **and** the contrast with traditional queries). The
  baseline's single lookup tends to favor whichever facet is lexically dominant and under-answers
  the other.
- **HyDE-at-ingestion improves matching.** Because each moment is indexed by the *questions it
  answers*, a user's natural-language question can match a stored question even when it shares few
  words with the raw transcript.
- **Hybrid + RRF balances meaning and keywords.** Dense vectors catch paraphrase; BM25 catches exact
  terms (e.g. "HNSW", "cosine"); RRF fuses them so neither dominates.
- **Cross-encoder re-rank sharpens precision.** It re-scores the fused shortlist jointly against the
  query, pushing the truly on-point moment to the top before synthesis.
- **Moments are coherent citeable units.** Semantic boundaries mean a cited moment is a self-contained
  span, not a fragment sliced mid-thought by a token budget.

**Tradeoffs.** Moment RAG costs more at ingestion (one LLM enrichment call per moment) and adds
query-time latency (decomposition + multi-branch retrieval + rerank). For a single video that's
negligible; at scale you'd cache enrichment (the reference commits it) and tune `top_n` / k.

## Appendix — Backing retrieval with a real vector database (ChromaDB)

Everything above uses an in-memory NumPy matrix as an *exact* similarity-search index. Here we
satisfy the **"vector database"** option of the assignment literally: we store the **same Part 1
baseline embeddings** in **ChromaDB** — a real vector DB with an HNSW (approximate-nearest-neighbor)
index and on-disk persistence — and query it.

We reuse the vectors already in `base_matrix`, so indexing adds **no extra OpenAI cost** (only the
one query embedding per question). Chroma returns a **distance** (`cosine distance = 1 − cosine
similarity`), so *lower = closer* — compare it to the cosine *scores* the NumPy baseline printed
in Part 1. On a corpus this small the results match the brute-force version; the difference only
shows up at scale, where Chroma's ANN index avoids comparing the query to every vector.

### Why NumPy by default (and ChromaDB only here)?

- **Transparency.** With NumPy the retrieval *is* the math — `sims = M @ q; np.argsort(-sims)[:k]` —
  so you can read exactly how a result is chosen. A vector DB hides that behind `collection.query(...)`.
- **The corpus is tiny.** 9 chunks, 10 moments, 39 question-vectors. Comparing a query against ~40
  vectors is instant with brute force; an HNSW index only pays off at thousands-to-millions of vectors.
- **Exact vs. approximate.** Brute-force cosine returns the *true* nearest neighbours. HNSW trades a
  little accuracy for speed — a great deal at scale, pointless for 40 vectors.
- **Hybrid is simpler in NumPy.** Moment RAG fuses three signals (dense text + dense questions + BM25)
  via RRF; keeping them as plain arrays + `rank_bm25` in one place is clearer than orchestrating
  multiple DB collections.
- **Zero infra.** No service, no index build, no persistence layer — NumPy is already a dependency.

**When you'd flip the default to a vector DB:** many vectors (a whole video library), a need for
persistence (don't re-embed each run), metadata filtering, or concurrent serving — exactly the
"Future improvements" in the README. The cell below shows that path with ChromaDB; the result here
**validates** the NumPy version (distance `0.268` ↔ cosine `0.732`, the same top chunk).

In [15]:
import chromadb

# In-memory client; swap to chromadb.PersistentClient(path="data/chroma") to persist on disk.
chroma = chromadb.EphemeralClient()
try:
    chroma.delete_collection("baseline_chunks")   # idempotent: wipe on re-run
except Exception:
    pass
collection = chroma.create_collection("baseline_chunks", metadata={"hnsw:space": "cosine"})

# Reuse the embeddings already computed in Part 1 (base_matrix) -> no extra OpenAI cost.
collection.add(
    ids=[c["chunk_id"] for c in base_chunks],
    embeddings=base_matrix.tolist(),
    documents=[c["text"] for c in base_chunks],
    metadatas=[{"start_ms": c["start_ms"], "start": fmt_ts(c["start_ms"])} for c in base_chunks],
)
print(f"stored {collection.count()} chunks in ChromaDB collection 'baseline_chunks'")

stored 9 chunks in ChromaDB collection 'baseline_chunks'


In [16]:
def baseline_rag_chroma(query: str, k: int = 4) -> dict:
    """Same baseline pipeline as Part 1, but retrieval is served by ChromaDB."""
    qv = embed_texts([query])[0].tolist()
    res = collection.query(query_embeddings=[qv], n_results=k)
    docs, metas, dists = res["documents"][0], res["metadatas"][0], res["distances"][0]

    sep = """

"""  # two newlines between context blocks
    context = sep.join(f"[{n+1}] {d}" for n, d in enumerate(docs))
    prompt = f"""Answer the question using ONLY the context below. If the answer isn't in the context, say so.

Context:
{context}

Question: {query}"""
    resp = client.chat.completions.create(
        model=LLM_MODEL, temperature=0.2,
        messages=[{"role": "user", "content": prompt}],
    )
    return {
        "answer": resp.choices[0].message.content,
        "chunks": [{"start": m["start"], "cosine_dist": round(d, 3), "text": doc[:160] + "…"}
                   for m, d, doc in zip(metas, dists, docs)],
    }

out = baseline_rag_chroma("How does a vector database find similar vectors?")
print(out["answer"])
print("--- retrieved from ChromaDB (cosine distance, lower = closer) ---")
for c in out["chunks"]:
    print(f'  [{c["start"]}] dist={c["cosine_dist"]}  {c["text"]}')

A vector database finds similar vectors by performing similarity searches as mathematical operations, looking for vector embeddings that are close to each other. It uses vector indexing with approximate nearest neighbor (ANN) algorithms to quickly find vectors that are very likely to be among the closest matches, rather than finding the exact closest match.
--- retrieved from ChromaDB (cosine distance, lower = closer) ---
  [7:53] dist=0.268  you can't effectively and efficiently compare your query vector to every single vector in the database. It would just be too slow. So there is a process to do t…
  [8:57] dist=0.33  Now, vector databases are a core feature of something called RAG, retrieval  augmented generation, where vector databases store chunks of documents and articles…
  [2:23] dist=0.349  similar items are positioned close together in vector space and dissimilar items are positioned far apart, and with vector databases, we can perform similarity …
  [6:45] dist=0.377  and t